# 1. Setup and Libraries

In [1]:
import os
import glob
import json
import numpy as np
import cv2
import mediapipe as mp
import pandas as pd
import concurrent.futures
from scipy.spatial.transform import Rotation
from pathlib import Path
from tqdm import tqdm
import math
import time

class OneEuroFilter:
    def __init__(self, t0, x0, dx0=0.0, min_cutoff=1.0, beta=0.0, d_cutoff=1.0):
        self.min_cutoff = float(min_cutoff)
        self.beta = float(beta)
        self.d_cutoff = float(d_cutoff)
        self.x_prev = x0
        self.dx_prev = np.zeros_like(x0) if isinstance(x0, np.ndarray) else float(dx0)
        self.t_prev = float(t0)

    def __call__(self, t, x):
        t_e = t - self.t_prev
        t_e = max(t_e, 1e-5)
        a_d = self.smoothing_factor(t_e, self.d_cutoff)
        dx = (x - self.x_prev) / t_e
        dx_hat = self.exponential_smoothing(a_d, dx, self.dx_prev)
        cutoff = self.min_cutoff + self.beta * np.linalg.norm(dx_hat)
        a = self.smoothing_factor(t_e, cutoff)
        x_hat = self.exponential_smoothing(a, x, self.x_prev)
        self.x_prev = x_hat
        self.dx_prev = dx_hat
        self.t_prev = t
        return x_hat

    def smoothing_factor(self, t_e, cutoff):
        r = 2 * math.pi * cutoff * t_e
        return r / (r + 1)

    def exponential_smoothing(self, a, x, x_prev):
        return a * x + (1 - a) * x_prev


class BoneLengthKinematics:
    def __init__(self):
        self.bone_lengths = {}
        self.alpha = 0.05
        self.parents = {
            13: 11, 15: 13, # L Arm
            14: 12, 16: 14, # R Arm
            25: 23, 27: 25, # L Leg
            26: 24, 28: 26  # R Leg
        }

    def __call__(self, pose_3d):
        if pose_3d is None:
            return pose_3d
            
        for child, parent in self.parents.items():
            dist = np.linalg.norm(pose_3d[child] - pose_3d[parent])
            if child not in self.bone_lengths:
                self.bone_lengths[child] = dist
            else:
                self.bone_lengths[child] = (1 - self.alpha) * self.bone_lengths[child] + self.alpha * dist
                
        corrected_pose = pose_3d.copy()
        for child in [13, 15, 14, 16, 25, 27, 26, 28]:
            parent = self.parents[child]
            target_length = self.bone_lengths[child]
            vec = corrected_pose[child] - corrected_pose[parent]
            length = np.linalg.norm(vec)
            if length > 1e-5:
                vec = vec / length
                corrected_pose[child] = corrected_pose[parent] + vec * target_length
                
        return corrected_pose


# 2. Dataset Directories

In [2]:
dataset_Folder = "G:/My Drive/GP Datasets/fit3d_data/"
file_path = dataset_Folder + 'fit3d_info.json'

# 3. Load FIT3D Metadata

In [3]:
info_JSON = pd.read_json(file_path, typ='series')
print("JSON loaded successfully as a series:")
print(info_JSON.head())

val_subj_names = ['s03', 's11']
all_train = [sub for sub in info_JSON['train_subj_names'] if sub not in val_subj_names]
all_camera_names = info_JSON['all_camera_names']

# Extract all actions from all subjects and find unique values
all_actions_list = [action for actions in info_JSON['subj_to_act'].values() for action in actions]
unique_actions = sorted(list(set(all_actions_list)))

print(f"Number of unique actions: {len(unique_actions)}")
print("Unique actions:")
print(unique_actions)

JSON loaded successfully as a series:
subj_to_act         {'s03': ['band_pull_apart', 'dumbbell_high_pul...
test_subj_names                                       [s02, s12, s13]
train_subj_names             [s03, s04, s05, s07, s08, s09, s10, s11]
all_camera_names             [50591643, 58860488, 60457274, 65906101]
dtype: object
Number of unique actions: 47
Unique actions:
['band_pull_apart', 'barbell_dead_row', 'barbell_row', 'barbell_shrug', 'burpees', 'clean_and_press', 'deadlift', 'diamond_pushup', 'drag_curl', 'dumbbell_biceps_curls', 'dumbbell_curl_trifecta', 'dumbbell_hammer_curls', 'dumbbell_high_pulls', 'dumbbell_overhead_shoulder_press', 'dumbbell_reverse_lunge', 'dumbbell_scaptions', 'man_maker', 'mule_kick', 'neutral_overhead_shoulder_press', 'one_arm_row', 'overhead_extension_thruster', 'overhead_trap_raises', 'pushup', 'side_lateral_raise', 'squat', 'standing_ab_twists', 'w_raise', 'walk_the_box', 'warmup_1', 'warmup_10', 'warmup_11', 'warmup_12', 'warmup_13', 'warmup_14

# 5. Mediapipe 3D Pose Evaluation
## 5.1 Configuration & Indices

In [4]:
# MediaPipe Pose setup
mp_pose = mp.solutions.pose

# Joint subset ordered by H36M joint index.
# Joint:        [ L_Hip, L_Knee, L_Ankle,  R_Hip, R_Knee, R_Ankle,  L_Shoulder, L_Elbow, L_Wrist, R_Shoulder, R_Elbow, R_Wrist]
fit3d_indices = [    1,      2,      3,     4,      5,       6,            11,     12,     13,          14,      15,       16]
mp_indices    = [   23,     25,      27,    24,     26,      28,           11,      13,      15,         12,      14,      16]

## 5.2 Utilities (Procrustes Alignment & Metric)

In [5]:
def procrustes_alignment_batched(pred, target):
    """
    Batched Procrustes alignment.
    pred, target: (num_frames, num_joints, 3)
    """
    mu_p = np.mean(pred, axis=1, keepdims=True)
    mu_t = np.mean(target, axis=1, keepdims=True)
    
    p_0 = pred - mu_p
    t_0 = target - mu_t
    
    norm_p = np.linalg.norm(p_0, axis=(1, 2), keepdims=True)
    norm_t = np.linalg.norm(t_0, axis=(1, 2), keepdims=True)
    norm_t = np.where(norm_t == 0, 1e-8, norm_t)
    norm_p = np.where(norm_p == 0, 1e-8, norm_p)
    
    p_0 = p_0 / norm_p
    t_0 = t_0 / norm_t
    
    # Kabsch algorithm (batched)
    H = np.matmul(p_0.transpose(0, 2, 1), t_0)
    U, S, Vt = np.linalg.svd(H)
    
    detR = np.linalg.det(np.matmul(Vt.transpose(0, 2, 1), U.transpose(0, 2, 1)))
    V_adj = Vt.transpose(0, 2, 1).copy()
    V_adj[detR < 0, :, 2] *= -1
    R = np.matmul(V_adj, U.transpose(0, 2, 1))
    
    aligned = np.matmul(p_0, R.transpose(0, 2, 1)) * norm_t + mu_t
    return aligned

def calculate_pa_mpjpe(pred_3d, gt_3d, mp_idx, gt_idx):
    """
    PA-MPJPE (Procrustes-Aligned MPJPE) — the primary evaluation metric.
    Removes the effect of global rotation, translation, and scale.
    pred_3d, gt_3d: full joint arrays in mm
    """
    num_frames = min(len(pred_3d), len(gt_3d))
    if num_frames == 0: return 0.0
    
    p_subset = np.array(pred_3d[:num_frames])[:, mp_idx]
    g_subset = np.array(gt_3d[:num_frames])[:, gt_idx]
    
    p_aligned = procrustes_alignment_batched(p_subset, g_subset)
    dist = np.linalg.norm(p_aligned - g_subset, axis=-1)
    return np.mean(dist)


# 6. Define MediaPipe 3D Processing Function

In [6]:
def process_frame_with_mediapipe(pose, image, best_box, last_pose):
    start_time = time.perf_counter()
    
    if best_box is not None:
        x1, y1, x2, y2 = best_box
        
        # Add margin
        margin_x = int((x2 - x1) * 0.1)
        margin_y = int((y2 - y1) * 0.1)
        
        h, w, _ = image.shape
        cx1 = max(0, x1 - margin_x)
        cy1 = max(0, y1 - margin_y)
        cx2 = min(w, x2 + margin_x)
        cy2 = min(h, y2 + margin_y)
        
        cropped_image = __import__("numpy").ascontiguousarray(image[cy1:cy2, cx1:cx2])
        
        # Run ONLY the model inference
        results = pose.process(cropped_image)
    else:
        results = pose.process(image)

    end_time = time.perf_counter()
    frame_time = end_time - start_time
    
    if results and getattr(results, 'pose_world_landmarks', None):
        world_lms = np.array([[lm.x, lm.y, lm.z] for lm in results.pose_world_landmarks.landmark])
        current_pose = world_lms
    else:
        current_pose = last_pose
        
    return current_pose, frame_time

def predict_mediapipe_3d_with_precomputed_bb(t, complexity):
    subj = t['subj']
    action = t['action']
    cam_id = t['cam_id']
    video_path = t['video_path']
    
    # Path to the bounding boxes you just generated
    bb_dir = Path(f"D:/GP/Pose/YOLO_Labels_Temp/train/{subj}/BB (Not Original)/{cam_id}/{action}")
    
    pred_3d_list = []
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None, 0.0
        
    W = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
    H = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
    
    total_inference_time = 0.0
    total_frames = 0
    frame_idx = 0
    
    mp_pose = mp.solutions.pose
    with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5, model_complexity=complexity, static_image_mode=False) as pose:
        last_pose = np.zeros((33, 3))
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break
                
            image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image.flags.writeable = False
            
            # Retrieve pre-calculated YOLO bb
            txt_file = bb_dir / f"frame_{frame_idx:06d}.txt"
            best_box = None
            if txt_file.exists() and txt_file.stat().st_size > 0:
                try:
                    line = txt_file.read_text().strip()
                    if line:
                        parts = line.split()
                        if len(parts) >= 5:
                            class_id, x_c, y_c, w, h = map(float, parts[:5])
                            x1 = int((x_c - w/2) * W)
                            y1 = int((y_c - h/2) * H)
                            x2 = int((x_c + w/2) * W)
                            y2 = int((y_c + h/2) * H)
                            best_box = (max(0, x1), max(0, y1), min(int(W), x2), min(int(H), y2))
                except:
                    pass
            
            current_pose, frame_time = process_frame_with_mediapipe(pose, image, best_box, last_pose)
            
            pred_3d_list.append(current_pose)
            last_pose = current_pose
            
            total_inference_time += frame_time
            total_frames += 1
            frame_idx += 1
                
    cap.release()
    average_fps = total_frames / total_inference_time if total_inference_time > 0 else 0.0
    if len(pred_3d_list) == 0:
        return None, average_fps
    return np.array(pred_3d_list), average_fps

def predict_mediapipe_3d_filtered(t, complexity):
    subj = t['subj']
    action = t['action']
    cam_id = t['cam_id']
    video_path = t['video_path']
    
    bb_dir = Path(f"D:/GP/Pose/YOLO_Labels_Temp/train/{subj}/BB (Not Original)/{cam_id}/{action}")
    
    pred_3d_list = []
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None, 0.0
        
    W = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
    H = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
    
    total_inference_time = 0.0
    total_frames = 0
    frame_idx = 0
    
    mp_pose = mp.solutions.pose
    filter_3d = None
    ik_solver = BoneLengthKinematics()
    
    with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5, model_complexity=complexity, static_image_mode=False) as pose:
        last_pose = np.zeros((33, 3))
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break
                
            image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image.flags.writeable = False
            
            txt_file = bb_dir / f"frame_{frame_idx:06d}.txt"
            best_box = None
            if txt_file.exists() and txt_file.stat().st_size > 0:
                try:
                    line = txt_file.read_text().strip()
                    if line:
                        parts = line.split()
                        if len(parts) >= 5:
                            class_id, x_c, y_c, w, h = map(float, parts[:5])
                            x1 = int((x_c - w/2) * W)
                            y1 = int((y_c - h/2) * H)
                            x2 = int((x_c + w/2) * W)
                            y2 = int((y_c + h/2) * H)
                            best_box = (max(0, x1), max(0, y1), min(int(W), x2), min(int(H), y2))
                except:
                    pass
            
            current_pose, frame_time = process_frame_with_mediapipe(pose, image, best_box, last_pose)
            
            if filter_3d is None:
                filter_3d = OneEuroFilter(t0=frame_time, x0=current_pose)
            else:
                current_pose = filter_3d(t=total_inference_time, x=current_pose)
            
            current_pose = ik_solver(current_pose)
            
            pred_3d_list.append(current_pose)
            last_pose = current_pose
            
            total_inference_time += frame_time
            total_frames += 1
            frame_idx += 1
                
    cap.release()
    average_fps = total_frames / total_inference_time if total_inference_time > 0 else 0.0
    if len(pred_3d_list) == 0:
        return None, average_fps
    return np.array(pred_3d_list), average_fps

def predict_mediapipe_3d_alone_filtered(t, complexity):
    subj = t['subj']
    action = t['action']
    cam_id = t['cam_id']
    video_path = t['video_path']
    
    pred_3d_list = []
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None, 0.0
        
    total_inference_time = 0.0
    total_frames = 0
    
    mp_pose = mp.solutions.pose
    filter_3d = None
    ik_solver = BoneLengthKinematics()
    
    with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5, model_complexity=complexity, static_image_mode=False) as pose:
        last_pose = np.zeros((33, 3))
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break
                
            image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image.flags.writeable = False
            
            best_box = None
            
            current_pose, frame_time = process_frame_with_mediapipe(pose, image, best_box, last_pose)
            
            if filter_3d is None:
                filter_3d = OneEuroFilter(t0=frame_time, x0=current_pose)
            else:
                current_pose = filter_3d(t=total_inference_time, x=current_pose)
            
            current_pose = ik_solver(current_pose)
            
            pred_3d_list.append(current_pose)
            last_pose = current_pose
            
            total_inference_time += frame_time
            total_frames += 1
                
    cap.release()
    average_fps = total_frames / total_inference_time if total_inference_time > 0 else 0.0
    if len(pred_3d_list) == 0:
        return None, average_fps
    return np.array(pred_3d_list), average_fps


# 7. Validation Evaluation Pipeline

## 7.1 Generate Validation Tasks

In [7]:
def get_validation_tasks(dataset_base, subj_names, all_camera_names):
    tasks = []
    base_path = Path(dataset_base)
    for subj in subj_names:
        ref_dir = base_path / "train" / subj / "joints3d_25"
        if not ref_dir.exists(): continue
        for ref_file in ref_dir.glob("*.json"):
            action = ref_file.stem
            gt_3d = None
            for cam_id in np.random.choice(all_camera_names, 1, replace=False):
                v_path = base_path / "train" / subj / "videos" / str(cam_id) / f"{action}.mp4"
                if v_path.exists():
                    if gt_3d is None:
                        with open(ref_file, 'r') as f:
                            gt_data = json.load(f)
                            gt_3d = np.array(gt_data.get('joints3d_25', list(gt_data.values())[0])) if isinstance(gt_data, dict) else np.array(gt_data)
                    tasks.append({'subj': subj, 'action': action, 'cam_id': cam_id, 'video_path': str(v_path), 'gt_3d': gt_3d})
    return tasks

## 7.2 Validation Execution Pipeline

In [8]:
def evaluate_validation_set(tasks, predict_func, complexity, mp_indices, fit3d_indices):
    print(f"Starting parallel validation evaluation of {len(tasks)} videos...", flush=True)
    all_pa_mpjpe = []
    all_fps = []
    action_pa_mpjpe = {}
    
    def process_task(t):
        result = predict_func(t, complexity)
        if result is None or result[0] is None: return None
        pred_3d_raw, average_fps = result
        
        gt_3d_mm   = t['gt_3d'] * 1000
        pred_3d_mm = pred_3d_raw * 1000.0
        try:
            pa = calculate_pa_mpjpe(pred_3d_mm, gt_3d_mm, mp_indices, fit3d_indices)
            return (t['subj'], t['action'], t['cam_id'], pa, average_fps)
        except Exception as e:
            return ('ERROR', t['subj'], t['action'], t.get('cam_id', 'TestCam'), e, 0.0)
        
    with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
        futures = [executor.submit(process_task, t) for t in tasks]
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(tasks), desc="Evaluating Pose"):
            res = future.result()
            if isinstance(res, tuple) and res[0] == 'ERROR':
                print(f"\nERROR processing {res[1]} | {res[2]} (Cam {res[3]}): {res[4]}")
            else:
                subj, action, cam_id, pa, average_fps = res
                all_pa_mpjpe.append(pa)
                action_pa_mpjpe.setdefault(action, []).append(pa)
                if average_fps > 0:
                    all_fps.append(average_fps)
                
    if all_pa_mpjpe:
        print("\n\n--- PA-MPJPE per Exercise ---")
        for act in sorted(action_pa_mpjpe.keys()):
            print(f"{act:15s}: PA-MPJPE = {np.mean(action_pa_mpjpe[act]):6.2f} mm")
        print(f"\nOverall Mean PA-MPJPE: {np.mean(all_pa_mpjpe):.2f} mm")
        if all_fps:
            print(f"Overall Mean FPS: {np.mean(all_fps):.2f} FPS")
    return all_pa_mpjpe


# 9. Execute Experiments (PA-MPJPE)

In [9]:
val_tasks = get_validation_tasks(dataset_Folder, val_subj_names, all_camera_names)

## Phase 1: Precomputed BB + MediaPipe 3D (complexity = 1)

### Experiment 1: BB (Not Original) + Medipipe 3D – complexity = 1

In [10]:
print("========== Experiment 1: Precomputed BB + Medipipe 3D – complexity = 1 ==========") 
_ = evaluate_validation_set(val_tasks, predict_mediapipe_3d_with_precomputed_bb, complexity=1, mp_indices=mp_indices, fit3d_indices=fit3d_indices)

========== Experiment 1: Precomputed BB + Medipipe 3D – complexity = 1 ==========
Starting parallel validation evaluation of 94 videos...


Evaluating Pose: 100%|██████████| 94/94 [58:48<00:00, 37.54s/it] 



--- PA-MPJPE per Exercise ---
band_pull_apart: PA-MPJPE =  63.41 mm
barbell_dead_row: PA-MPJPE =  91.92 mm
barbell_row    : PA-MPJPE =  82.50 mm
barbell_shrug  : PA-MPJPE =  77.33 mm
burpees        : PA-MPJPE = 124.20 mm
clean_and_press: PA-MPJPE =  80.36 mm
deadlift       : PA-MPJPE =  75.07 mm
diamond_pushup : PA-MPJPE = 136.10 mm
drag_curl      : PA-MPJPE =  92.52 mm
dumbbell_biceps_curls: PA-MPJPE =  64.02 mm
dumbbell_curl_trifecta: PA-MPJPE =  65.54 mm
dumbbell_hammer_curls: PA-MPJPE =  63.87 mm
dumbbell_high_pulls: PA-MPJPE =  65.53 mm
dumbbell_overhead_shoulder_press: PA-MPJPE =  75.59 mm
dumbbell_reverse_lunge: PA-MPJPE =  71.69 mm
dumbbell_scaptions: PA-MPJPE =  62.13 mm
man_maker      : PA-MPJPE =  90.32 mm
mule_kick      : PA-MPJPE = 131.45 mm
neutral_overhead_shoulder_press: PA-MPJPE =  73.12 mm
one_arm_row    : PA-MPJPE =  97.39 mm
overhead_extension_thruster: PA-MPJPE =  72.80 mm
overhead_trap_raises: PA-MPJPE =  80.23 mm
pushup         : PA-MPJPE =  85.38 mm
side_later

## Phase 2: MediaPipe 3D + One Euro Filter & IK (No BB) (complexity = 1)

### Experiment 2: MediaPipe 3D alone + One Euro Filter + Inverse Kinematics

In [11]:
print("========== Experiment 2: MediaPipe 3D alone + One Euro + IK ==========")
_ = evaluate_validation_set(val_tasks, predict_mediapipe_3d_alone_filtered, complexity=1, mp_indices=mp_indices, fit3d_indices=fit3d_indices)

========== Experiment 2: MediaPipe 3D alone + One Euro + IK ==========
Starting parallel validation evaluation of 94 videos...


Evaluating Pose: 100%|██████████| 94/94 [1:28:13<00:00, 56.32s/it] 



--- PA-MPJPE per Exercise ---
band_pull_apart: PA-MPJPE =  63.80 mm
barbell_dead_row: PA-MPJPE =  92.04 mm
barbell_row    : PA-MPJPE =  82.94 mm
barbell_shrug  : PA-MPJPE =  77.27 mm
burpees        : PA-MPJPE = 124.93 mm
clean_and_press: PA-MPJPE =  80.40 mm
deadlift       : PA-MPJPE =  74.76 mm
diamond_pushup : PA-MPJPE = 132.57 mm
drag_curl      : PA-MPJPE =  92.52 mm
dumbbell_biceps_curls: PA-MPJPE =  64.06 mm
dumbbell_curl_trifecta: PA-MPJPE =  65.67 mm
dumbbell_hammer_curls: PA-MPJPE =  63.80 mm
dumbbell_high_pulls: PA-MPJPE =  65.68 mm
dumbbell_overhead_shoulder_press: PA-MPJPE =  75.02 mm
dumbbell_reverse_lunge: PA-MPJPE =  71.54 mm
dumbbell_scaptions: PA-MPJPE =  62.27 mm
man_maker      : PA-MPJPE =  89.77 mm
mule_kick      : PA-MPJPE = 129.09 mm
neutral_overhead_shoulder_press: PA-MPJPE =  72.80 mm
one_arm_row    : PA-MPJPE =  97.50 mm
overhead_extension_thruster: PA-MPJPE =  72.60 mm
overhead_trap_raises: PA-MPJPE =  79.69 mm
pushup         : PA-MPJPE =  85.04 mm
side_later

## Phase 3: Precomputed BB + MediaPipe 3D + One Euro Filter & IK (complexity = 1)

### Experiment 3: Precomputed BB + MediaPipe 3D + One Euro Filter + Inverse Kinematics

In [12]:
print("========== Experiment 3: Precomputed BB + Medipipe 3D + One Euro + IK ==========")
_ = evaluate_validation_set(val_tasks, predict_mediapipe_3d_filtered, complexity=1, mp_indices=mp_indices, fit3d_indices=fit3d_indices)

========== Experiment 3: Precomputed BB + Medipipe 3D + One Euro + IK ==========
Starting parallel validation evaluation of 94 videos...


Evaluating Pose: 100%|██████████| 94/94 [1:23:06<00:00, 53.04s/it]



--- PA-MPJPE per Exercise ---
band_pull_apart: PA-MPJPE =  63.73 mm
barbell_dead_row: PA-MPJPE =  92.02 mm
barbell_row    : PA-MPJPE =  82.94 mm
barbell_shrug  : PA-MPJPE =  77.28 mm
burpees        : PA-MPJPE = 123.99 mm
clean_and_press: PA-MPJPE =  80.53 mm
deadlift       : PA-MPJPE =  74.81 mm
diamond_pushup : PA-MPJPE = 132.43 mm
drag_curl      : PA-MPJPE =  92.53 mm
dumbbell_biceps_curls: PA-MPJPE =  64.09 mm
dumbbell_curl_trifecta: PA-MPJPE =  65.69 mm
dumbbell_hammer_curls: PA-MPJPE =  63.79 mm
dumbbell_high_pulls: PA-MPJPE =  65.68 mm
dumbbell_overhead_shoulder_press: PA-MPJPE =  75.06 mm
dumbbell_reverse_lunge: PA-MPJPE =  71.55 mm
dumbbell_scaptions: PA-MPJPE =  62.27 mm
man_maker      : PA-MPJPE =  90.27 mm
mule_kick      : PA-MPJPE = 129.24 mm
neutral_overhead_shoulder_press: PA-MPJPE =  72.78 mm
one_arm_row    : PA-MPJPE =  97.49 mm
overhead_extension_thruster: PA-MPJPE =  72.61 mm
overhead_trap_raises: PA-MPJPE =  79.68 mm
pushup         : PA-MPJPE =  85.04 mm
side_later